In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q12/q12_rewrite/checkpoints/pre_cell_2.pickle

In [ ]:
%%cudf.pandas.profile
### cell 2 ###
# 1) filter lineitem on GPU using string literals to avoid CPU Timestamps
flineitem = lineitem[
    lineitem.L_RECEIPTDATE.between("1994-01-01", "1995-01-01", inclusive='left')
    & (lineitem.L_COMMITDATE  < "1995-01-01")
    & (lineitem.L_SHIPDATE   < "1995-01-01")
    & (lineitem.L_SHIPDATE   < lineitem.L_COMMITDATE)
    & (lineitem.L_COMMITDATE < lineitem.L_RECEIPTDATE)
    & lineitem.L_SHIPMODE.isin(["MAIL", "SHIP"])
][["L_ORDERKEY", "L_SHIPMODE"]]

# 2) project only the columns we need from orders
small_orders = orders[["O_ORDERKEY", "O_ORDERPRIORITY"]]

# 3) join on minimal columns
jn = flineitem.merge(
    small_orders,
    left_on="L_ORDERKEY",
    right_on="O_ORDERKEY"
)

# 4) build urgency flags on GPU
jn["is_urgent"]     = jn.O_ORDERPRIORITY.isin(["1-URGENT", "2-HIGH"])
jn["is_not_urgent"] = ~jn["is_urgent"]

# 5) group and aggregate in one pass
total = jn.groupby("L_SHIPMODE", as_index=False).agg(
    g1=("is_urgent",     "sum"),
    g2=("is_not_urgent", "sum")
)

print(total)